In [ ]:
# Cell 1: TPU Environment Setup & Repository Cloning
import os
import sys
import subprocess

print("🚀 Setting up TPU v5e-8 environment...")
repo_url = "https://github.com/dsainvg001/transformer-math.git"
repo_dir = "transformer-math"

if not os.path.exists(repo_dir):
    print(f"Cloning repository from {repo_url}...")
    subprocess.run(["git", "clone", repo_url], check=True)
else:
    print(f"Repository {repo_dir} already exists.")

if os.path.exists(repo_dir):
    os.chdir(repo_dir)
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

print(f"Current Working Directory: {os.getcwd()}")
import jax
devices = jax.devices()
print(f"JAX Backend: {jax.default_backend()}")
print(f"JAX TPU Devices Detected ({len(devices)}): {devices}")

In [ ]:
# Cell 2: Configuration & Debug Mode Toggle for TPU v5e-8
DEBUG_MODE = False  # Set to True for a fast 1-minute dry run test

if DEBUG_MODE:
    print("=== RUNNING IN TPU DEBUG MODE (Fast 1-minute Pipeline Test) ===")
    config = {
        "seed": 42,
        "precision": "bfloat16",
        "model": {
            "context_len": 64,
            "num_layers": 2,
            "num_heads": 4,
            "emb_dim": 128,
            "mlp_dim": 512,
            "pos_emb_type": "sinusoidal"
        },
        "data": {
            "max_depth": 2,
            "float_precision": 1,
            "enabled_ops": ["+", "-", "*", "/"],
            "val_size": 64,
            "test_size": 64
        },
        "training": {
            "total_steps": 10,
            "warmup_steps": 2,
            "learning_rate": 0.001,
            "batch_size": 64,
            "max_grad_norm": 1.0,
            "weight_decay": 0.01,
            "eval_interval": 5,
            "save_interval": 10,
            "log_interval": 1,
            "checkpoint_dir": "/kaggle/working/checkpoints_tpu_debug",
            "accuracy_epsilon": 0.05,
            "target_accuracy": 0.999
        },
        "curriculum": {
            "enabled": True,
            "ema_alpha": 0.1,
            "temperature": 0.5,
            "floor_prob": 0.005,
            "overfit_ratio": 3.0,
            "overfit_train_threshold": 0.15,
            "overfit_decay": 0.1,
            "update_interval": 5
        }
    }
else:
    print("=== RUNNING IN PRODUCTION MODE (TPU v5e-8 Multi-Core Training ~25.25M Params) ===")
    config = {
        "seed": 42,
        "precision": "bfloat16",
        "model": {
            "context_len": 64,
            "num_layers": 8,
            "num_heads": 8,
            "emb_dim": 512,
            "mlp_dim": 2048,
            "pos_emb_type": "sinusoidal"
        },
        "data": {
            "max_depth": 3,
            "float_precision": 1,
            "enabled_ops": ["+", "-", "*", "/", "^", "sin", "cos", "tan", "log", "ln", "exp", "sqrt", "abs"],
            "val_size": 512,
            "test_size": 8192
        },
        "training": {
            "total_steps": 50000,
            "warmup_steps": 2000,
            "learning_rate": 0.0005,
            "batch_size": 512,
            "max_grad_norm": 1.0,
            "weight_decay": 0.01,
            "eval_interval": 500,
            "save_interval": 5000,
            "log_interval": 50,
            "checkpoint_dir": "/kaggle/working/checkpoints_tpu",
            "accuracy_epsilon": 0.01,
            "target_accuracy": 0.999
        },
        "curriculum": {
            "enabled": True,
            "ema_alpha": 0.1,
            "temperature": 0.5,
            "floor_prob": 0.005,
            "overfit_ratio": 3.0,
            "overfit_train_threshold": 0.15,
            "overfit_decay": 0.1,
            "update_interval": 250
        }
    }

In [ ]:
# Cell 3: Live Plotter Helper Class
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

class LivePlotter:
    def __init__(self):
        self.train_steps = []
        self.train_losses = []
        self.val_steps = []
        self.val_losses = []
        self.val_exact_matches = []
        self.val_tolerant_accs = []

    def update_train(self, step: int, loss: float):
        self.train_steps.append(step)
        self.train_losses.append(loss)

    def update_val(self, step: int, loss: float, exact_match: float, tolerant_acc: float):
        self.val_steps.append(step)
        self.val_losses.append(loss)
        self.val_exact_matches.append(exact_match * 100.0)
        self.val_tolerant_accs.append(tolerant_acc * 100.0)

    def plot(self):
        try:
            # clear_output(wait=True) # Commented out to preserve console logs
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
            
            if self.train_steps:
                ax1.plot(self.train_steps, self.train_losses, label="Train Loss", color="dodgerblue", alpha=0.8, linewidth=1.5)
            if self.val_steps:
                ax1.plot(self.val_steps, self.val_losses, label="Val Loss", color="crimson", marker="o", linewidth=2.0)
            ax1.set_xlabel("Step")
            ax1.set_ylabel("Cross Entropy Loss")
            ax1.set_title("Training & Validation Loss (On The Go)")
            ax1.grid(True, linestyle="--", alpha=0.5)
            ax1.legend()

            if self.val_steps:
                ax2.plot(self.val_steps, self.val_exact_matches, label="Exact Match %", color="green", marker="s", linewidth=2.0)
                ax2.plot(self.val_steps, self.val_tolerant_accs, label="Tolerant Acc %", color="darkorange", marker="^", linewidth=2.0)
            ax2.set_xlabel("Step")
            ax2.set_ylabel("Accuracy (%)")
            ax2.set_title("Validation Accuracy Metrics")
            ax2.set_ylim(-5, 105)
            ax2.grid(True, linestyle="--", alpha=0.5)
            ax2.legend()

            plt.tight_layout()
            display(plt.gcf())
            plt.close(fig)
        except Exception:
            pass

In [ ]:
# Cell 4: Initialize Data Sampler, Model, and TPU Training State
import time
import json
import numpy as np
import jax.numpy as jnp
import optax
from flax.training import train_state
import flax.jax_utils as jutils

from src.tokenizer.tokenizer import Tokenizer
from src.data.sampler import ExpressionSampler
from src.data.curriculum import CurriculumTracker
from src.model.transformer import TransformerDecoder
from src.train import make_parallel_train_step, train_step, CheckpointManager, RobustCheckpointManager, CustomTrainState
from src.eval import evaluate_on_dataset

seed = config["seed"]
devices = jax.devices()
num_devices = len(devices)
batch_size = config["training"]["batch_size"]
per_device_batch = batch_size // num_devices if num_devices > 0 else batch_size
context_len = config["model"]["context_len"]
total_steps = config["training"]["total_steps"]

tokenizer = Tokenizer()
sampler = ExpressionSampler(
    tokenizer=tokenizer,
    max_depth=config["data"]["max_depth"],
    float_precision=config["data"]["float_precision"],
    context_len=context_len,
    seed=seed,
    enabled_ops=config["data"]["enabled_ops"],
    val_size=config["data"]["val_size"],
    test_size=config["data"]["test_size"]
)

cur_cfg = config["curriculum"]
cur_tracker = CurriculumTracker(
    categories=sampler.categories,
    ema_alpha=cur_cfg.get("ema_alpha", 0.1),
    temperature=cur_cfg.get("temperature", 0.5),
    floor_prob=cur_cfg.get("floor_prob", 0.005),
    overfit_ratio=cur_cfg.get("overfit_ratio", 3.0),
    overfit_train_threshold=cur_cfg.get("overfit_train_threshold", 0.15),
    overfit_decay=cur_cfg.get("overfit_decay", 0.1)
)

model = TransformerDecoder(
    vocab_size=tokenizer.vocab_size,
    context_len=context_len,
    num_layers=config["model"]["num_layers"],
    num_heads=config["model"]["num_heads"],
    emb_dim=config["model"]["emb_dim"],
    mlp_dim=config["model"]["mlp_dim"],
    pos_emb_type=config["model"].get("pos_emb_type", "sinusoidal"),
    dtype=jnp.bfloat16 if config["precision"] == "bfloat16" else jnp.float32,
    param_dtype=jnp.bfloat16 if config["precision"] == "bfloat16" else jnp.float32
)

init_rng = jax.random.PRNGKey(seed)
dummy_inputs = jnp.ones((1, context_len - 1), dtype=jnp.int32)
variables = model.init(init_rng, dummy_inputs)
params = variables["params"]

learning_rate = config["training"]["learning_rate"]
warmup_steps = config["training"].get("warmup_steps", 1000)
schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=learning_rate,
    warmup_steps=warmup_steps,
    decay_steps=total_steps,
    end_value=learning_rate * 0.01
)

tx = optax.chain(
    optax.clip_by_global_norm(config["training"].get("max_grad_norm", 1.0)),
    optax.adamw(learning_rate=schedule, weight_decay=config["training"].get("weight_decay", 0.01))
)

state = CustomTrainState.create(
    apply_fn=model.apply,
    params=params,
    tx=tx
)

if num_devices > 1:
    print(f"✅ TPU Multi-Core JAX pmap ACTIVE: Running on {num_devices} TPU cores with per-device batch size {per_device_batch}")
    replicated_state = jutils.replicate(state)
    parallel_step = make_parallel_train_step(model)
else:
    print(f"Running on single device: {devices[0]}")
    replicated_state = None
    parallel_step = None

checkpoint_manager = RobustCheckpointManager(config["training"]["checkpoint_dir"], max_to_keep=3)
print("TPU initialization complete. Ready to launch training!")

In [ ]:
# Cell 5: TPU Training Loop
plotter = LivePlotter()
start_time = time.time()
cur_enabled = cur_cfg.get("enabled", True)

train_stream = sampler.stream_batches(
    batch_size=batch_size,
    get_probs_fn=cur_tracker.get_probabilities if cur_enabled else None,
    prefetch_size=32
)

log_interval = config["training"].get("log_interval", 50)
eval_interval = config["training"].get("eval_interval", 500)
save_interval = config["training"].get("save_interval", 5000)
target_accuracy = config["training"].get("target_accuracy", 0.999)

print(f"🚀 Starting TPU Training Loop ({total_steps} steps total, batch size {batch_size})...", flush=True)

for step in range(1, total_steps + 1):
    if step == 1:
        print("⏳ Step 1: Compiling TPU XLA Graph across 8 cores (takes ~2-3 mins on first step)...", flush=True)
        
    batch = next(train_stream)
    
    if num_devices > 1:
        batch_reshaped = {
            "input_ids": batch["input_ids"].reshape(num_devices, per_device_batch, -1),
            "loss_mask": batch["loss_mask"].reshape(num_devices, per_device_batch, -1),
            "category_idx": batch["category_idx"].reshape(num_devices, per_device_batch)
        }
        replicated_state, loss_arr, seq_losses_arr = parallel_step(
            replicated_state,
            batch_reshaped["input_ids"],
            batch_reshaped["loss_mask"]
        )
        loss_val = float(loss_arr[0])
        seq_losses_val = np.array(seq_losses_arr[0]).flatten()
        cat_idxs = batch["category_idx"]
    else:
        state, loss_val_raw, seq_losses_val = train_step(state, batch)
        loss_val = float(loss_val_raw)
        cat_idxs = batch["category_idx"]
        
    if cur_enabled:
        for cat_idx, seq_loss in zip(cat_idxs, seq_losses_val):
            cur_tracker.update_train_loss(int(cat_idx), float(seq_loss))
        if step % cur_cfg.get("update_interval", 250) == 0:
            cur_tracker.recompute_weights()
            
    if step % log_interval == 0 or step == 1:
        print(f"Step {step}/{total_steps} | Loss: {loss_val:.4f} | Samples: {step*batch_size:,}", flush=True)
        
    if step % eval_interval == 0 or step == total_steps:
        current_state = jutils.unreplicate(replicated_state) if num_devices > 1 else state
        eval_metrics = evaluate_on_dataset(
            model=model,
            params=current_state.params,
            tokenizer=tokenizer,
            dataset=sampler.val_set,
            context_len=config["model"]["context_len"],
            epsilon=config["training"].get("accuracy_epsilon", 0.05),
            max_eval_samples=128
        )
        
        val_loss = eval_metrics["overall/loss"]
        exact_match = eval_metrics["overall/exact_match"]
        tolerant_acc = eval_metrics["overall/tolerant_accuracy"]
        
        # plotter.plot() # Commented out to preserve console logs
        
        if cur_enabled:
            val_losses_to_feed = {cat: eval_metrics[f"category_loss/{cat}"] for cat in sampler.categories if f"category_loss/{cat}" in eval_metrics}
            cur_tracker.update_val_losses(val_losses_to_feed)
            
        print(f"Step {step}/{total_steps} | Val Loss: {val_loss:.4f} | Exact Match: {exact_match*100:.2f}% | Tolerant Acc: {tolerant_acc*100:.2f}% | Samples: {step*batch_size:,}", flush=True)
        
        if not DEBUG_MODE and tolerant_acc >= target_accuracy and step >= 2000:
            print(f"🎯 Reached target accuracy threshold ({tolerant_acc*100:.2f}% >= {target_accuracy*100:.0f}%). Stopping training early!", flush=True)
            current_state = jutils.unreplicate(replicated_state) if num_devices > 1 else state
            checkpoint_manager.save(step, current_state)
            break

    if step % save_interval == 0 or step == total_steps:
        current_state = jutils.unreplicate(replicated_state) if num_devices > 1 else state
        checkpoint_manager.save(step, current_state)
        
print("🎉 TPU Training process finished successfully!")

In [ ]:
# Cell 6: Save TPU Model Checkpoints & Dataset Shards
print("💾 Saving Final Model & Dataset Shards...")
dataset_save_dir = "/kaggle/working/saved_dataset_tpu" if not DEBUG_MODE else "./saved_dataset_tpu_debug"
os.makedirs(dataset_save_dir, exist_ok=True)

final_state = jutils.unreplicate(replicated_state) if num_devices > 1 else state
checkpoint_manager.save(step if 'step' in locals() else total_steps, final_state)
print(f"💾 Saved Model Checkpoints to: {config['training']['checkpoint_dir']}")

print("💾 Exporting generated dataset shards...")
sampler.dump_offline_dataset(
    output_dir=dataset_save_dir,
    num_examples=1000 if DEBUG_MODE else 10000,
    shard_size=1000 if DEBUG_MODE else 5000
)

val_file = os.path.join(dataset_save_dir, "val_set.json")
test_file = os.path.join(dataset_save_dir, "test_set.json")

with open(val_file, "w") as f:
    json.dump(sampler.val_set, f, indent=2)

with open(test_file, "w") as f:
    json.dump(sampler.test_set, f, indent=2)

print(f"✅ Saved offline dataset shards to: {dataset_save_dir}")
print(f"   - Sharded JSONL files created: {len(os.listdir(dataset_save_dir))}")
print(f"   - Validation set saved: {val_file}")
print(f"   - Test set saved: {test_file}")
checkpoint_manager.close()

In [ ]:
# Cell 7: Independent Final Testing & Data Corruption Audit on TPU
from src.eval import verify_dataset_integrity

print("===============================================================")
print("🔍 TPU INDEPENDENT DATASET INTEGRITY & DATA LEAKAGE AUDIT REPORT")
print("===============================================================")
integrity_report = verify_dataset_integrity(
    test_set=sampler.test_set,
    held_out_exprs=sampler.held_out_exprs,
    seen_train_exprs=sampler.seen_train_exprs
)
print(f"- Total Independent Test Samples: {integrity_report['total_test_samples']}")
print(f"- Training Set Overlap (Data Leakage): {integrity_report['leaked_samples']} samples ({integrity_report['leakage_rate_pct']:.2f}% Leakage Rate)")
print(f"- Target Formatting Corruption: {integrity_report['corrupted_samples']} samples ({integrity_report['corruption_rate_pct']:.2f}% Corruption Rate)")
status_str = "✅ 100% UNCORRUPTED & INDEPENDENT" if integrity_report['is_data_clean'] else "❌ INTEGRITY ISSUE DETECTED"
print(f"- Data Integrity Status: {status_str}")

print("\n===============================================================")
print("🎯 FINAL INDEPENDENT TEST SET EVALUATION")
print("===============================================================")
final_eval_state = jutils.unreplicate(replicated_state) if num_devices > 1 else state
test_metrics = evaluate_on_dataset(
    model=model,
    params=final_eval_state.params,
    tokenizer=tokenizer,
    dataset=sampler.test_set,
    context_len=config["model"]["context_len"],
    epsilon=config["training"].get("accuracy_epsilon", 0.05)
)
print(f"- Independent Test Loss: {test_metrics['overall/loss']:.4f}")
print(f"- Independent Test Exact-Match: {test_metrics['overall/exact_match']*100:.2f}%")
print(f"- Independent Test Tolerant Acc: {test_metrics['overall/tolerant_accuracy']*100:.2f}%")
print("===============================================================")